# Stage 2 — NER Pipeline
**Goal:** Extract person names from every manifesto using CamemBERT-NER.

By the end of this notebook you will have:
- A citation dataset: every person name mentioned in every manifesto
- Each citation linked to the candidate's gender, year, and party
- A quick evaluation of the NER quality on a sample

## 0. Setup

In [11]:
import pandas as pd
from pathlib import Path
from transformers import pipeline
from tqdm import tqdm

PROJECT_ROOT = Path("..")
ELECTION_YEARS = ["1973", "1978", "1981", "1988", "1993"]

## 1. Reload the joined dataset from Stage 1

We re-run the minimal loading code from notebook 01 to get `df` (manifestos + gender labels).

In [12]:
TEXT_ROOT = PROJECT_ROOT / "data" / "archelec_repo" / "text_files"
METADATA_PATH = PROJECT_ROOT / "data" / "archelec_metadata_full.csv"

# Load texts
records = []
for year in ELECTION_YEARS:
    folder = TEXT_ROOT / year / "legislatives"
    for fpath in sorted(folder.glob("*.txt")):
        try:
            text = fpath.read_text(encoding="utf-8", errors="replace")
        except Exception:
            text = ""
        records.append({"year": year, "candidate_id": fpath.stem, "text": text})
df_manifestos = pd.DataFrame(records)

# Load and join metadata
df_meta = pd.read_csv(METADATA_PATH, sep=",", encoding="utf-8", low_memory=False)
df_meta_leg = df_meta[df_meta["subject"].str.contains("législatives", na=False)].copy()
df_meta_leg["year"] = pd.to_datetime(df_meta_leg["date"]).dt.year.astype(str)
df_meta_leg = df_meta_leg[df_meta_leg["year"].isin(ELECTION_YEARS)]

df = df_manifestos.merge(
    df_meta_leg[["id", "titulaire-sexe", "titulaire-soutien", "titulaire-nom", "titulaire-prenom", "year"]],
    left_on="candidate_id", right_on="id", how="left"
)

# Keep only matched rows with known gender
df = df[df["titulaire-sexe"].isin(["homme", "femme"])].reset_index(drop=True)
print(f"Manifestos to process: {len(df):,}")
print(df["titulaire-sexe"].value_counts())

Manifestos to process: 20,331
titulaire-sexe
homme    18171
femme     2160
Name: count, dtype: int64


## 2. Load CamemBERT-NER

We use `Jean-Baptiste/camembert-ner` — CamemBERT fine-tuned on French NER datasets (WikiNER + French TreeBank).
It recognizes 4 entity types: **PER** (person), LOC (location), ORG (organization), MISC (miscellaneous).

We only care about **PER** — the person names mentioned in manifestos.

In [13]:
ner = pipeline(
    "ner",
    model="Jean-Baptiste/camembert-ner",
    aggregation_strategy="simple",  # merges token pieces into full entity spans
    device=0 #-1  # CPU; change to 0 if you have a GPU
)
print("Model loaded.")

Device set to use cuda:0


Model loaded.


## 3. Test on one manifesto

Before running on 21k texts, always test on one example.
This lets you check: are the extracted names sensible? Is the format what you expect?

In [14]:
sample_text = df["text"].iloc[0]
sample_candidate = df["candidate_id"].iloc[0]
print(f"Candidate: {sample_candidate}")
print(f"Text preview: {sample_text[:300]}")
print("\n--- NER output ---")

entities = ner(sample_text[:1000])  # test on first 1000 chars
persons = [e for e in entities if e["entity_group"] == "PER"]
for p in persons:
    print(f"  {p['word']!r:30s}  score={p['score']:.2f}")

Candidate: EL065_L_1973_03_001_01_1_PF_01
Text preview: Sciences Po / fonds CEVIPOF
REPUBLIQUE FRANÇAISE - LIBERTE - EGALITE - FRATERNITE
Paul BARBEROT
Département de l'AIN 1re Circonscription (BOURG-EN-BRESSE)
ÉLECTIONS LÉGISLATIVES du 4 MARS 1973
CENTRE - PROGRES ET DEMOCRATIE MODERNE
investi par L'UNION DES RÉPUBLICAINS DE PROGRÈS pour le soutien au P

--- NER output ---
  'Paul BARBEROT'                 score=0.99
  'Président de la République'    score=0.75


## 4. Run NER on all manifestos

**Important:** CamemBERT has a 512-token limit. Most manifestos are ~500 words which is fine.
For longer ones we truncate — we lose the tail of the text but keep the most important part
(candidates typically cite figures early in the manifesto).

This will take a few minutes on CPU.

In [17]:
def extract_persons(text, max_chars=2000):
    if not text or len(text.strip()) == 0:
        return []
    entities = ner(text[:max_chars])
    return [e["word"] for e in entities if e["entity_group"] == "PER" and e["score"] > 0.7]

citations = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Running NER"):
    persons = extract_persons(row["text"])
    year = row["year_x"] if "year_x" in row.index else row["year"]
    for person in persons:
        citations.append({
            "candidate_id":    row["candidate_id"],
            "candidate_sex":   row["titulaire-sexe"],
            "candidate_party": row["titulaire-soutien"],
            "candidate_name":  f"{row['titulaire-prenom']} {row['titulaire-nom']}",
            "year":            year,
            "cited_person":    person,
        })

df_citations = pd.DataFrame(citations)
print(f"\nTotal citations extracted: {len(df_citations):,}")
print(f"Unique persons cited: {df_citations['cited_person'].nunique():,}")
df_citations.head(10)

Running NER: 100%|██████████| 20331/20331 [10:52<00:00, 31.18it/s]


Total citations extracted: 85,827
Unique persons cited: 38,460


,candidate_id,candidate_sex,candidate_party,candidate_name,year,cited_person
0,EL065_L_1973_03_001_01_1_PF_01,homme,Centre progrès et démocratie moderne;Union des...,Paul Barberot,1973,Paul BARBEROT
1,EL065_L_1973_03_001_01_1_PF_01,homme,Centre progrès et démocratie moderne;Union des...,Paul Barberot,1973,Président de la République
2,EL065_L_1973_03_001_01_1_PF_01,homme,Centre progrès et démocratie moderne;Union des...,Paul Barberot,1973,Jean-Marie BEAUDET
3,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,Roland MON NET
4,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,Pierre PERRIN
5,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,Roland MONNET
6,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,PIERRE PERRIN
7,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,ROLAND MONNET
8,EL065_L_1973_03_001_01_1_PF_02,homme,Parti socialiste;Mouvement des radicaux de gauche,Roland Monnet,1973,Pierre PERRIN
9,EL065_L_1973_03_001_01_1_PF_03,homme,non mentionné,Michel Chomarat,1973,MICHEL CHOMARAT


## 5. Save the citation dataset

In [18]:
out_path = PROJECT_ROOT / "data" / "citations.csv"
df_citations.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to ../data/citations.csv


## 6. Quick evaluation — top cited figures

Before moving to the gender analysis, sanity-check the results:
are the most cited figures actual politicians? Do they make historical sense?

In [19]:
print("=== Top 30 most cited persons overall ===")
print(df_citations["cited_person"].value_counts().head(30).to_string())

=== Top 30 most cited persons overall ===
cited_person
François MITTERRAND           2092
Président de la République    2015
François Mitterrand           1429
Mademoiselle                   765
Mitterrand                     646
Chirac                         618
Barre                          603
Giscard                        531
Monsieur                       511
Michel ROCARD                  484
Jean-Marie LE PEN              454
Arlette Laguiller              400
CHIRAC                         397
BARRE                          347
LE PEN                         336
Général de Gaulle              310
Michel Rocard                  310
Jacques CHIRAC                 273
Arlette LAGUILLER              247
Président                      230
Le Pen                         198
Giscard d'Estaing              194
FRANÇOIS MITTERRAND            183
Brice LALONDE                  180
Madame                         176
MITTERRAND                     172
Premier Ministre               159


In [20]:
print("=== Top 20 cited by FEMALE candidates ===")
female_cit = df_citations[df_citations["candidate_sex"] == "femme"]
print(female_cit["cited_person"].value_counts().head(20).to_string())

print("\n=== Top 20 cited by MALE candidates ===")
male_cit = df_citations[df_citations["candidate_sex"] == "homme"]
print(male_cit["cited_person"].value_counts().head(20).to_string())

=== Top 20 cited by FEMALE candidates ===
cited_person
François MITTERRAND           212
François Mitterrand           211
Barre                         174
Chirac                        173
Arlette Laguiller             166
Mitterrand                    164
Président de la République    155
DASSAULT                      143
Giscard                       117
CHIRAC                        108
BARRE                         100
Arlette LAGUILLER              92
DE WENDEL                      69
Jean-Marie LE PEN              65
Mademoiselle                   61
Michel ROCARD                  58
Brice LALONDE                  47
Monsieur                       45
Gisèle HALIMI                  40
Jean Rostand                   40

=== Top 20 cited by MALE candidates ===
cited_person
François MITTERRAND           1880
Président de la République    1860
François Mitterrand           1218
Mademoiselle                   704
Mitterrand                     482
Monsieur                       466
C

In [21]:
print("=== Citations per year ===")
year_col = "year_x" if "year_x" in df_citations.columns else "year"
print(df_citations.groupby([year_col, "candidate_sex"]).size().unstack(fill_value=0))

=== Citations per year ===
candidate_sex  femme  homme
year                       
1973             548  12719
1978            3044  17059
1981            1592  14185
1988            1823  16109
1993            2092  16656


## 7. Summary

By the end of this notebook you should have:
- [ ] `data/citations.csv` — the full citation dataset
- [ ] Top cited figures make historical sense (De Gaulle, Mitterrand, Giscard...)
- [ ] Female and male candidates show different citation patterns

**Next:** `03_gender_tagging.ipynb` — query Wikidata to label the gender of each cited figure.